## Transfer Volume

This notebook copies export artifacts from the source volume to the target import volume.

**What it does:**
1. Creates the import volume if it doesn't exist
2. Copies all files from the export volume to the import volume
3. Verifies file count and integrity

**Prerequisites:**
- Source and target catalogs on the same UC metastore
- READ access to source export volume
- WRITE access to target import volume

## Read Parameters

In [ ]:
import os
import shutil

dbutils.widgets.text("source_volume_path", "", "Source Export Volume")
dbutils.widgets.text("target_volume_path", "", "Target Import Volume")
dbutils.widgets.text("target_catalog", "", "Target Catalog")
dbutils.widgets.text("target_schema", "", "Target Schema")
dbutils.widgets.text("import_volume", "", "Import Volume Name")

source_path = dbutils.widgets.get("source_volume_path")
target_path = dbutils.widgets.get("target_volume_path")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
import_volume = dbutils.widgets.get("import_volume")

if not source_path:
    raise ValueError("source_volume_path parameter is required")
if not target_path:
    raise ValueError("target_volume_path parameter is required")

print(f"Source volume: {source_path}")
print(f"Target volume: {target_path}")

## Create Import Volume if Needed

In [ ]:
if target_catalog and target_schema and import_volume:
    create_sql = f"CREATE VOLUME IF NOT EXISTS {target_catalog}.{target_schema}.{import_volume}"
    print(f"Creating volume: {create_sql}")
    spark.sql(create_sql)
    print("✓ Volume ready")
else:
    print("Skipping volume creation (parameters not provided)")

## Verify Source Volume

In [ ]:
if not os.path.exists(source_path):
    raise FileNotFoundError(
        f"Source volume not found: {source_path}\n"
        "Ensure the export job completed and the metastore is shared."
    )

def count_files(path):
    """Recursively count files in a directory."""
    total = 0
    for root, dirs, files in os.walk(path):
        total += len(files)
    return total

source_file_count = count_files(source_path)
print(f"Source volume: {source_path}")
print(f"Files to transfer: {source_file_count}")

## Transfer Files

In [ ]:
os.makedirs(target_path, exist_ok=True)

print("Transferring files...")

transferred = 0
errors = []

for root, dirs, files in os.walk(source_path):
    rel_root = os.path.relpath(root, source_path)
    target_dir = os.path.join(target_path, rel_root) if rel_root != "." else target_path
    
    os.makedirs(target_dir, exist_ok=True)
    
    for filename in files:
        src_file = os.path.join(root, filename)
        tgt_file = os.path.join(target_dir, filename)
        
        try:
            shutil.copy2(src_file, tgt_file)
            transferred += 1
        except Exception as e:
            errors.append({"file": src_file, "error": str(e)})

print(f"\nTransferred: {transferred} files")
if errors:
    print(f"Errors: {len(errors)}")
    for err in errors:
        print(f"  - {err['file']}: {err['error']}")

## Verify Transfer

In [ ]:
target_file_count = count_files(target_path)

print(f"\nVerification:")
print(f"  Source files:  {source_file_count}")
print(f"  Target files:  {target_file_count}")

if target_file_count >= source_file_count:
    print("\n✓ Transfer complete")
else:
    print(f"\n⚠️  Missing {source_file_count - target_file_count} file(s)")

## List Transferred Artifacts

In [ ]:
print("\nTransferred artifacts:")
for item in os.listdir(target_path):
    item_path = os.path.join(target_path, item)
    if os.path.isdir(item_path):
        subcount = count_files(item_path)
        print(f"  {item}/ ({subcount} files)")
    else:
        print(f"  {item}")

## Summary

In [ ]:
print("=" * 60)
print("TRANSFER COMPLETE")
print("=" * 60)
print(f"Files transferred: {transferred}")
print(f"Target volume:     {target_path}")
print("\nNext step: Run Tgt_02_Deploy_Genie_Spaces to deploy the spaces.")